In [ ]:
import sys, os
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0,str(ROOT))
sys.path.insert(0, str(ROOT / 'src'))

from src.splits import get_cv_splits
from src.eval import evaluate_and_log

In [2]:
os.environ['WANDB_ENTITY'] = 'emammadli'

In [3]:
P = ROOT / 'data' / 'processed'
train = pd.read_csv(P / 'train_clean.csv')
folds = pd.read_csv(P / 'folds.csv')
train = train.merge(folds[['record', 'minute', 'patient_fold', 'random_fold']], on=['record', 'minute'], validate='1:1')

print(train.shape)

BASE = ['rr_mean', 'rr_sdnn', 'rr_rmssd', 'rr_pnn50', 'rr_min', 'rr_max', 'rr_range']
Z = [c + '_z' for c in BASE]
CTX = [c + '_ctx' for c in Z]
SETS = {'A': BASE, 'B': Z, 'C': Z + CTX, 'D': BASE + Z + CTX}




(16531, 27)


In [ ]:
def run_ml_cv(model, feat_set, split_col='patient_fold', run_name=None, use_wandb=False):
    feats = SETS[feat_set]
    oof = np.zeros(len(train))
    for tr, va in get_cv_splits(train, split_col):
        m = clone(model).fit(train.iloc[tr][feats], train.iloc[tr]['label'])
        oof[va] = m.predict_proba(train.iloc[va][feats])[:, 1]

    split_type = 'patient_fold' if split_col == 'patient_fold' else 'random_fold'
    metrics = evaluate_and_log(train, oof, model_name=f'{run_name}_{feat_set}',
                               split_type=split_type,
                               config={'family': 'ML','model': run_name, 'features': feat_set, 'n_features': len(feats)},
                               use_wandb=use_wandb, wandb_project='apnea-ecg-ml')
    return oof, metrics

In [76]:
res = []
def run_and_store(model, name, sets=('A', 'B', 'C', 'D'), split_col = 'patient_fold', use_wandb = False):
    for s in sets:
        _, m = run_ml_cv(model=model, feat_set=s, split_col=split_col,run_name=name,use_wandb=use_wandb)
        res.append({'model': name, 'set': s, 'split': split_col, **m})

In [77]:
lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
dtree = DecisionTreeClassifier(max_depth=6, min_samples_leaf=1, min_samples_split=2)
rf = RandomForestClassifier(n_estimators=400, min_samples_leaf=3,n_jobs = -1, random_state=42)
xgbt = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.3, random_state=42,n_jobs = -1)

MODELS = {
    'lr':lr,
    'dtree':dtree,
    'rf':rf,
    'xgb':xgbt,
}

for i, j in MODELS.items():
    run_and_store(j,i, use_wandb=False)

[patient_fold | lr_A] Acc=0.6846 Sens=0.7303 Spec=0.6549 F1=0.6459 AUC=0.7288 (thr=0.35, PatAcc=0.686)
[patient_fold | lr_B] Acc=0.6145 Sens=0.8309 Spec=0.4740 F1=0.6294 AUC=0.6993 (thr=0.32, PatAcc=0.657)
[patient_fold | lr_C] Acc=0.6578 Sens=0.8171 Spec=0.5543 F1=0.6529 AUC=0.7395 (thr=0.33, PatAcc=0.657)
[patient_fold | lr_D] Acc=0.7392 Sens=0.7027 Spec=0.7629 F1=0.6797 AUC=0.7787 (thr=0.40, PatAcc=0.686)
[patient_fold | dtree_A] Acc=0.6687 Sens=0.7587 Spec=0.6102 F1=0.6434 AUC=0.6838 (thr=0.35, PatAcc=0.657)
[patient_fold | dtree_B] Acc=0.5903 Sens=0.8820 Spec=0.4007 F1=0.6291 AUC=0.6945 (thr=0.18, PatAcc=0.657)
[patient_fold | dtree_C] Acc=0.6439 Sens=0.8372 Spec=0.5183 F1=0.6494 AUC=0.7262 (thr=0.29, PatAcc=0.657)
[patient_fold | dtree_D] Acc=0.6637 Sens=0.6764 Spec=0.6555 F1=0.6131 AUC=0.6765 (thr=0.34, PatAcc=0.629)
[patient_fold | rf_A] Acc=0.6265 Sens=0.7810 Spec=0.5261 F1=0.6222 AUC=0.6972 (thr=0.23, PatAcc=0.657)
[patient_fold | rf_B] Acc=0.6455 Sens=0.8108 Spec=0.5380 F1=0

In [78]:
results = pd.DataFrame(res)
results.pivot_table(index='model', columns='set', values='roc_auc').round(3)

set,A,B,C,D
model,,,,
dtree,0.684,0.695,0.726,0.676
lr,0.729,0.699,0.739,0.779
rf,0.697,0.742,0.786,0.741
xgb,0.690,0.710,0.762,0.707


In [70]:
pd.DataFrame(res).to_csv(ROOT / 'reports' / 'ml_init_results_patient.csv', index=False)

In [74]:
for i, j in MODELS.items():
    run_and_store(j,i, sets=('C', 'D'), split_col='random_fold', use_wandb=True)

accuracy,▁
f1,▁
patient_frac_mae,▁
patient_screen_acc,▁
roc_auc,▁
sensitivity,▁
specificity,▁
threshold,▁
accuracy,0.65138
f1,0.65543
patient_frac_mae,0.31871


[random_fold | lr_C] Acc=0.6514 Sens=0.8418 Spec=0.5276 F1=0.6554 AUC=0.7467 (thr=0.32, PatAcc=0.657)


accuracy,▁
f1,▁
patient_frac_mae,▁
patient_screen_acc,▁
roc_auc,▁
sensitivity,▁
specificity,▁
threshold,▁
accuracy,0.7861
f1,0.73769
patient_frac_mae,0.17872


[random_fold | lr_D] Acc=0.7861 Sens=0.7636 Spec=0.8007 F1=0.7377 AUC=0.8591 (thr=0.42, PatAcc=0.714)


accuracy,▁
f1,▁
patient_frac_mae,▁
patient_screen_acc,▁
roc_auc,▁
sensitivity,▁
specificity,▁
threshold,▁
accuracy,0.68127
f1,0.68103
patient_frac_mae,0.27676


[random_fold | dtree_C] Acc=0.6813 Sens=0.8639 Spec=0.5626 F1=0.6810 AUC=0.7716 (thr=0.28, PatAcc=0.657)


... (6 outputs left out)

accuracy,▁
f1,▁
patient_frac_mae,▁
patient_screen_acc,▁
roc_auc,▁
sensitivity,▁
specificity,▁
threshold,▁
accuracy,0.83582
f1,0.80265
patient_frac_mae,0.09713


[random_fold | xgb_C] Acc=0.8358 Sens=0.8476 Spec=0.8281 F1=0.8026 AUC=0.9157 (thr=0.38, PatAcc=0.714)


accuracy,▁
f1,▁
patient_frac_mae,▁
patient_screen_acc,▁
roc_auc,▁
sensitivity,▁
specificity,▁
threshold,▁
accuracy,0.9017
f1,0.87535
patient_frac_mae,0.03966


[random_fold | xgb_D] Acc=0.9017 Sens=0.8764 Spec=0.9182 F1=0.8754 AUC=0.9608 (thr=0.47, PatAcc=0.971)


In [72]:
results = pd.DataFrame(res)
results.pivot_table(index='model', columns='set', values='roc_auc').round(3)

set,A,B,C,D
model,,,,
dtree,0.688,0.694,0.749,0.758
lr,0.729,0.699,0.743,0.819
rf,0.697,0.742,0.830,0.844
xgb,0.690,0.710,0.839,0.834
